# Model Definitions

This notebook defines all model architectures, evaluation metrics, and cross-validation
schemes required by the thesis. It exposes reusable functions that Notebook 5 calls
to run the full experiment matrix.

**Environment:** `ml_env`

## Contents

1. Load experiment data from Notebook 3
2. Metric functions (Pearson r, MSE, F1, accuracy, probabilistic Pearson r)
3. Model factory (creates fresh model instances with consistent hyperparameters)
4. Cross-validation schemes (Group K-Fold / LOPO, K-Fold K=18)
5. Single-experiment runner (trains one model on one feature set with one CV scheme)
6. Full experiment runner (iterates over all combinations)
7. Quick smoke test to verify everything works

In [1]:
import numpy as np
import pickle
import os
import warnings
from time import time

from scipy.stats import pearsonr

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score

warnings.filterwarnings("ignore", category=UserWarning)  # suppress sklearn convergence warnings during CV

## 1. Load Experiment Data

Load the consolidated pickle from Notebook 3.

In [2]:
with open("features/experiment_data.pkl", "rb") as f:
    data = pickle.load(f)

feature_sets    = data["feature_sets"]
targets         = data["targets"]
participant_ids = data["participant_ids"]
component_dims  = data["component_dims"]
sfa_variants    = data["sfa_variants"]
feature_label   = data["feature_label"]

print(f"Feature sets:    {len(feature_sets)}")
print(f"Component dims:  {component_dims}")
print(f"SFA variants:    {sfa_variants}")
print(f"Feature label:   {feature_label}")
print(f"Participants:    {len(np.unique(participant_ids))}")

Feature sets:    7
Component dims:  [5, 10, 15, 20, 25, 30]
SFA variants:    ['slow_linear', 'slow_deg2', 'slow_deg3']
Feature label:   audio
Participants:    18


## 2. Evaluation Metrics

As specified in the thesis outline:

**Regression:**
- Pearson's correlation coefficient
- Mean squared error

**Classification:**
- F1 score
- Accuracy
- Pearson's correlation coefficient using the probabilistic outputs of the models

In [3]:
def regression_metrics(y_true, y_pred):
    """
    Compute regression metrics.

    Returns dict with:
        pearson_r:  Pearson correlation coefficient
        mse:        Mean squared error
    """
    if np.std(y_pred) < 1e-10:
        # Constant predictions — Pearson r is undefined
        r = 0.0
    else:
        r, _ = pearsonr(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    return {"pearson_r": r, "mse": mse}


def classification_metrics(y_true, y_pred, y_prob):
    """
    Compute classification metrics.

    Args:
        y_true: Ground truth binary labels
        y_pred: Predicted binary labels
        y_prob: Predicted probability for the positive class

    Returns dict with:
        f1:         Weighted F1 score
        accuracy:   Classification accuracy
        pearson_r:  Pearson r between true labels and probabilistic outputs
    """
    f1 = f1_score(y_true, y_pred, average="weighted")
    acc = accuracy_score(y_true, y_pred)

    if np.std(y_prob) < 1e-10:
        r = 0.0
    else:
        r, _ = pearsonr(y_true, y_prob)

    return {"f1": f1, "accuracy": acc, "pearson_r": r}

## 3. Model Factory

A factory function that returns a fresh, untrained model instance.
This ensures identical architectures across all feature sets,
as required by the outline (slow feature models must match PCA baseline architectures).

In [4]:
def create_model(model_name):
    """
    Create a fresh model instance by name.

    Regression models:
        'linear_regression'  — LinearRegression
        'mlp_regressor'      — MLPRegressor(100, 50)

    Classification models:
        'logistic_regression' — LogisticRegression(max_iter=1000)
        'mlp_classifier'      — MLPClassifier(100, 50)
    """
    models = {
        "linear_regression": lambda: LinearRegression(),
        "mlp_regressor": lambda: MLPRegressor(
            hidden_layer_sizes=(100, 50),
            max_iter=500,
            random_state=42,
        ),
        "logistic_regression": lambda: LogisticRegression(
            max_iter=1000,
            random_state=42,
        ),
        "mlp_classifier": lambda: MLPClassifier(
            hidden_layer_sizes=(100, 50),
            max_iter=500,
            random_state=42,
        ),
    }
    if model_name not in models:
        raise ValueError(f"Unknown model: {model_name}. Choose from {list(models.keys())}")
    return models[model_name]()


REGRESSION_MODELS = ["linear_regression", "mlp_regressor"]
CLASSIFICATION_MODELS = ["logistic_regression", "mlp_classifier"]

## 4. Cross-Validation Schemes

Two CV schemes as specified in the outline:

1. **Group K-Fold (Leave-One-Participant-Out)**
   - Each participant = one group
   - K = number of participants (18 for RECOLA)
   - Tests cross-participant generalisation

2. **K-Fold (K=18, shuffled)**
   - Data from all participants shuffled together
   - K = 18
   - Tests general model performance (samples from same participant can appear in both train and test)

In [5]:
def get_cv_splitter(cv_scheme, participant_ids):
    """
    Return a cross-validation splitter and its name.

    Args:
        cv_scheme:       'group_kfold' or 'kfold'
        participant_ids: array of participant IDs (needed for group_kfold)

    Returns:
        (splitter, n_splits, groups_or_none)
    """
    n_participants = len(np.unique(participant_ids))

    if cv_scheme == "group_kfold":
        # Leave-One-Participant-Out: K = number of participants
        splitter = GroupKFold(n_splits=n_participants)
        return splitter, n_participants, participant_ids

    elif cv_scheme == "kfold":
        # Standard K-Fold with K=18, shuffled
        splitter = KFold(n_splits=n_participants, shuffle=True, random_state=42)
        return splitter, n_participants, None

    else:
        raise ValueError(f"Unknown CV scheme: {cv_scheme}")


CV_SCHEMES = ["group_kfold", "kfold"]

## 5. Single Experiment Runner

Trains and evaluates one model on one feature set using one CV scheme.
Returns per-fold train and test metrics.

In [6]:
def run_single_experiment(X, y, model_name, task, cv_scheme, participant_ids):
    """
    Run a single experiment: one model, one feature set, one target, one CV scheme.

    Args:
        X:               Feature matrix (n_samples, n_features)
        y:               Target array
        model_name:      One of REGRESSION_MODELS or CLASSIFICATION_MODELS
        task:            'regression' or 'classification'
        cv_scheme:       'group_kfold' or 'kfold'
        participant_ids: Participant ID array

    Returns:
        List of dicts, one per fold, each containing:
            'fold':          Fold index
            'train_metrics': Dict of metric_name -> value
            'test_metrics':  Dict of metric_name -> value
    """
    splitter, n_splits, groups = get_cv_splitter(cv_scheme, participant_ids)
    fold_results = []

    split_iter = (
        splitter.split(X, y, groups=groups)
        if groups is not None
        else splitter.split(X, y)
    )

    for fold_idx, (train_idx, test_idx) in enumerate(split_iter):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Scale features per fold (fit on train, transform both)
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # Create and train model
        model = create_model(model_name)
        model.fit(X_train, y_train)

        # Compute metrics
        if task == "regression":
            y_pred_train = model.predict(X_train)
            y_pred_test = model.predict(X_test)
            train_metrics = regression_metrics(y_train, y_pred_train)
            test_metrics = regression_metrics(y_test, y_pred_test)

        elif task == "classification":
            y_pred_train = model.predict(X_train)
            y_pred_test = model.predict(X_test)
            y_prob_train = model.predict_proba(X_train)[:, 1]
            y_prob_test = model.predict_proba(X_test)[:, 1]
            train_metrics = classification_metrics(y_train, y_pred_train, y_prob_train)
            test_metrics = classification_metrics(y_test, y_pred_test, y_prob_test)

        fold_results.append({
            "fold": fold_idx,
            "train_metrics": train_metrics,
            "test_metrics": test_metrics,
        })

    return fold_results

## 6. Full Experiment Runner

Iterates over all combinations of:
- Feature sets (raw, PCA, slow linear, slow degree 2, slow degree 3)
- Targets (arousal, valence)
- Models (linear regression, MLP regressor, logistic regression, MLP classifier)
- CV schemes (Group K-Fold, K-Fold)

This function is designed to be called from Notebook 5.
It can also be called here with a subset of feature sets for quick testing.

In [7]:
def run_all_experiments(feature_sets, targets, participant_ids,
                        cv_schemes=CV_SCHEMES, verbose=True):
    """
    Run the full experiment matrix.

    Args:
        feature_sets:    Dict of name -> X array
        targets:         Dict of task -> target_name -> y array
        participant_ids: Participant ID array
        cv_schemes:      List of CV scheme names
        verbose:         Print progress

    Returns:
        List of result dicts, each containing:
            'feature_set':  Name of the feature set
            'target':       Target name (arousal/valence)
            'task':         Task type (regression/classification)
            'model':        Model name
            'cv_scheme':    CV scheme name
            'fold_results': List of per-fold metrics
    """
    all_results = []

    # Build experiment list
    experiments = []
    for feat_name in sorted(feature_sets.keys()):
        for task, task_targets in targets.items():
            model_list = REGRESSION_MODELS if task == "regression" else CLASSIFICATION_MODELS
            for target_name, y in task_targets.items():
                for model_name in model_list:
                    for cv_scheme in cv_schemes:
                        experiments.append((
                            feat_name, task, target_name, y,
                            model_name, cv_scheme
                        ))

    total = len(experiments)
    if verbose:
        print(f"Total experiments to run: {total}")
        print(f"(Feature sets: {len(feature_sets)} x Targets: 2 x "
              f"Models: {len(REGRESSION_MODELS)+len(CLASSIFICATION_MODELS)} x "
              f"CV schemes: {len(cv_schemes)})\n")

    for i, (feat_name, task, target_name, y, model_name, cv_scheme) in enumerate(experiments):
        X = feature_sets[feat_name]

        if verbose:
            print(f"[{i+1}/{total}] {feat_name:25s} | {target_name:8s} | "
                  f"{model_name:22s} | {cv_scheme:12s}", end="")

        t0 = time()
        fold_results = run_single_experiment(
            X, y, model_name, task, cv_scheme, participant_ids
        )
        elapsed = time() - t0

        # Compute mean test metric for quick summary
        mean_metrics = {}
        for key in fold_results[0]["test_metrics"]:
            vals = [fr["test_metrics"][key] for fr in fold_results]
            mean_metrics[key] = np.mean(vals)

        if verbose:
            summary = " | ".join(f"{k}={v:.4f}" for k, v in mean_metrics.items())
            print(f" | {elapsed:5.1f}s | {summary}")

        all_results.append({
            "feature_set": feat_name,
            "target": target_name,
            "task": task,
            "model": model_name,
            "cv_scheme": cv_scheme,
            "fold_results": fold_results,
        })

    if verbose:
        print(f"\nDone. {len(all_results)} experiments completed.")

    return all_results

## 7. Results Save/Load Helpers

Utility functions for persisting results to disk.

In [8]:
def save_results(results, filename="results/experiment_results.pkl"):
    """Save experiment results to a pickle file."""
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, "wb") as f:
        pickle.dump(results, f)
    size_mb = os.path.getsize(filename) / (1024 * 1024)
    print(f"Saved {len(results)} experiment results to {filename} ({size_mb:.1f} MB)")


def load_results(filename="results/experiment_results.pkl"):
    """Load experiment results from a pickle file."""
    with open(filename, "rb") as f:
        results = pickle.load(f)
    print(f"Loaded {len(results)} experiment results from {filename}")
    return results

## 8. Smoke Test

Run a minimal subset to verify everything works end-to-end before
committing to the full experiment matrix in Notebook 5.

We test with just the `raw` features, one target (arousal),
and both CV schemes.

In [ ]:
# Run a quick smoke test with raw features only
smoke_features = {"raw": feature_sets["raw"]}

smoke_targets = {
    "regression": {"arousal": targets["regression"]["arousal"]},
    "classification": {"arousal": targets["classification"]["arousal"]},
}

smoke_results = run_all_experiments(
    smoke_features, smoke_targets, participant_ids, verbose=True
)

print(f"\nSmoke test passed: {len(smoke_results)} experiments completed.")

Total experiments to run: 8
(Feature sets: 1 x Targets: 2 x Models: 4 x CV schemes: 2)

[1/8] raw                       | arousal  | linear_regression      | group_kfold  |   3.3s | pearson_r=0.5193 | mse=0.0293
[2/8] raw                       | arousal  | linear_regression      | kfold        |   3.1s | pearson_r=0.5391 | mse=0.0256
[3/8] raw                       | arousal  | mlp_regressor          | group_kfold  | 186.4s | pearson_r=0.3851 | mse=0.0367
[4/8] raw                       | arousal  | mlp_regressor          | kfold        | 207.7s | pearson_r=0.6714 | mse=0.0216
[5/8] raw                       | arousal  | logistic_regression    | group_kfold  |  13.9s | f1=0.6949 | accuracy=0.6790 | pearson_r=0.4458
[6/8] raw                       | arousal  | logistic_regression    | kfold        |  14.2s | f1=0.7209 | accuracy=0.7210 | pearson_r=0.4914
[7/8] raw                       | arousal  | mlp_classifier         | group_kfold  | 458.7s | f1=0.6456 | accuracy=0.6304 | pearson_r=

### Inspect smoke test results

Verify the per-fold structure is correct.

In [ ]:
# Show detailed results for one experiment
example = smoke_results[0]
print(f"Feature set:  {example['feature_set']}")
print(f"Target:       {example['target']}")
print(f"Task:         {example['task']}")
print(f"Model:        {example['model']}")
print(f"CV scheme:    {example['cv_scheme']}")
print(f"Num folds:    {len(example['fold_results'])}")

print(f"\nPer-fold test metrics:")
for fr in example["fold_results"]:
    metrics_str = " | ".join(f"{k}={v:.4f}" for k, v in fr["test_metrics"].items())
    print(f"  Fold {fr['fold']:2d}: {metrics_str}")

# Show mean and std
print(f"\nMean +/- std (test):")
for key in example["fold_results"][0]["test_metrics"]:
    vals = [fr["test_metrics"][key] for fr in example["fold_results"]]
    print(f"  {key:12s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")

## Usage from Notebook 5

Notebook 5 can import and run the full experiment matrix like this:

```python
# Option A: Run this notebook first, then Notebook 5 in the same session
#           (all functions will be in memory)

# Option B: Copy the function definitions into Notebook 5
#           and load experiment_data.pkl there

# Full run:
results = run_all_experiments(feature_sets, targets, participant_ids)
save_results(results)
```

### Experiment counts

With all 25 feature sets loaded:
- 25 feature sets x 2 targets x 2 regression models x 2 CV schemes = **200 regression experiments**
- 25 feature sets x 2 targets x 2 classification models x 2 CV schemes = **200 classification experiments**
- **Total: 400 experiments**, each with 18 folds